In [11]:
import pandas as pd
import numpy as np

raw = pd.read_csv("dataset.csv")
print('first 5 rows:')
print(raw.head(), end='\n\n')
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))
print('Datatypes of columns;',raw.dtypes)
print('Basic data analysis details: ',raw.describe().T)


first 5 rows:
  Drone ID                Application Drone Size    Drone Model  Manufacturer  \
0     D001           Package Delivery     Medium    FlyHigh 300      AeroCorp   
1     D002  Infrastructure Inspection      Large     Inspecta X  SkyView Inc.   
2     D003      Agricultural Spraying      Large     CropMaster    AgriDrones   
3     D004         Aerial Photography      Small  SnapShot Mini      PhotoFly   
4     D005               Surveillance     Medium    Watcher Pro    SecureTech   

   Propeller Count  Max Carry Weight (kg)  Actual Carry Weight (kg)  \
0              4.0                    5.0                       2.5   
1              6.0                   15.0                       8.0   
2              8.0                   20.0                      18.0   
3              4.0                    1.0                       0.5   
4              4.0                    7.0                       3.0   

     Payload Type                       Payload Description  ...  \
0   

In [12]:
print("Missing Values Count:")
print(raw.isnull().sum())

print(f"\nNumber of Duplicate Rows: {raw.duplicated().sum()}")

for col in raw.select_dtypes(include=["object"]).columns:
  print(f"\nUnique values in '{col}':", raw[col].unique())

print("\n--- Outlier Detection (IQR Method) ")
num_cols = raw.select_dtypes(include=[np.number]).columns
for col in num_cols:
  Q1 = raw[col].quantile(0.25)
  Q3 = raw[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR #Interquantile range: its a logic that applies the difference between the medians of the top and bottom fourth
  upper_bound = Q3 + 1.5 * IQR
  outliers = raw[(raw[col] < lower_bound) | (raw[col] > upper_bound)]
  print(f"Column '{col}': {len(outliers)} potential outliers detected.")

Missing Values Count:
Drone ID                       4
Application                    4
Drone Size                     4
Drone Model                    4
Manufacturer                   4
Propeller Count                4
Max Carry Weight (kg)          4
Actual Carry Weight (kg)       6
Payload Type                   4
Payload Description            4
Altitude (meters)              4
Flight Duration (minutes)      6
Distance Flown (km)            4
Operator ID                    4
Flight Date                    4
Battery Remaining (%)          4
GPS Accuracy (meters)          4
Wind Speed (m/s)               4
Obstacles Encountered          4
Flight Status                  4
Regulatory Approval ID         4
Notes                        502
dtype: int64

Number of Duplicate Rows: 22

Unique values in 'Drone ID': <StringArray>
['D001', 'D002', 'D003', 'D004', 'D005', 'D006', 'D007', 'D008', 'D009',
 'D010',
 ...
 'D744', 'D745', 'D746', 'D747', 'D748', 'D819', 'D749', 'D750', 'D751',
 'D75

/tmp/ipykernel_36895/3947582005.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in raw.select_dtypes(include=["object"]).columns:


In [13]:
cleaned = raw.drop_duplicates().copy()

for col in num_cols:
  if cleaned[col].isnull().sum() > 0:
    cleaned[col] = cleaned[col].fillna(cleaned[col].median()) #imputes average values for missing, i think this is what u wanted
    
cat_cols = cleaned.select_dtypes(include=["object"]).columns
for col in cat_cols:
  if cleaned[col].isnull().sum() > 0:
    cleaned[col] = cleaned[col].fillna(cleaned[col].mode()[0])

cleaned.to_csv("cleaned_drone_data.csv", index=False)
print("Dataset successfully cleaned and saved as 'cleaned_drone_data.csv'")
print(f"Cleaned dataset shape: {cleaned.shape}")

Dataset successfully cleaned and saved as 'cleaned_drone_data.csv'
Cleaned dataset shape: (534, 22)


/tmp/ipykernel_36895/2317535814.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = cleaned.select_dtypes(include=["object"]).columns


In [14]:
print("Minimum values:")
print(cleaned[num_cols].min(), end="\n\n")

print("Maximum values:")
print(cleaned[num_cols].max(), end="\n\n")

print("Average (Mean) values:")
print(cleaned[num_cols].mean(), end="\n\n")

if len(cat_cols) > 0:
  print(f"Value frequencies for categorical column '{cat_cols[0]}':")
  print(cleaned[cat_cols[0]].value_counts(), end="\n\n")

  if len(num_cols) > 0:
    print(f"Average '{num_cols[0]}' grouped by '{cat_cols[0]}':") #avg propeller count grouped by drone id 
    print(cleaned.groupby(cat_cols[0])[num_cols[0]].mean())

Minimum values:
Propeller Count               4.0
Max Carry Weight (kg)         1.0
Actual Carry Weight (kg)     -0.2
Altitude (meters)            10.0
Flight Duration (minutes)    10.0
Distance Flown (km)           2.0
Battery Remaining (%)        35.0
GPS Accuracy (meters)         0.5
Wind Speed (m/s)              1.0
dtype: float64

Maximum values:
Propeller Count                 8.0
Max Carry Weight (kg)          20.0
Actual Carry Weight (kg)       25.0
Altitude (meters)            8444.0
Flight Duration (minutes)      72.0
Distance Flown (km)            28.0
Battery Remaining (%)         100.0
GPS Accuracy (meters)           4.0
Wind Speed (m/s)               12.9
dtype: float64

Average (Mean) values:
Propeller Count               5.007491
Max Carry Weight (kg)         7.786330
Actual Carry Weight (kg)      5.787079
Altitude (meters)            89.739700
Flight Duration (minutes)    30.101124
Distance Flown (km)           9.207491
Battery Remaining (%)        74.000000
GPS Accura